# InvoiceGuard Round 2 -- Training Reproducibility Notebook

**Hackathon:** OpenEnv Hackathon 2026 (Round 2)
**Author:** piyush-mk
**Model:** Qwen/Qwen3-4B-Instruct-2507 (4-bit LoRA)

This notebook reproduces the Round 2 training pipeline for InvoiceGuard. It trains an open-weight LLM agent to solve accounts-payable exception resolution tasks using two methods:

1. **SFT (Supervised Fine-Tuning)** -- learns from expert traces generated by the environment's ground truth.
2. **GRPO (Group Relative Policy Optimization)** -- reinforcement learning with trajectory-level rewards from the environment grader.

The notebook pulls training code from `piyush-mk/invoiceguard-code` on Hugging Face Hub and pushes trained adapters + training artifacts back to the Hub. No local checkout is required.

**Outputs:** LoRA adapter repo + `training_artifacts/` containing `metrics.jsonl`, rollout samples, summary JSON, and PNG reward curves.

## Prerequisites

**GPU:** A100, L4, L40S, or any GPU with at least 24 GB VRAM. Qwen3-4B uses 4-bit quantization + LoRA + gradient checkpointing.

**Secrets:** Set `HF_TOKEN` with write access to the Hugging Face Hub.
- **Colab:** Settings > Secrets > add `HF_TOKEN`
- **Kaggle:** Add-ons > Secrets > add `HF_TOKEN`, then attach it to the notebook

**No local clone needed.** Training code is pulled from `piyush-mk/invoiceguard-code` on the Hub.

In [ ]:
# Install the same runtime style used by Hugging Face Jobs.
%pip -q install -U uv huggingface_hub


In [ ]:
import os

# Colab: store HF_TOKEN in Secrets as HF_TOKEN.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
except Exception:
    pass

# Kaggle: add HF_TOKEN under Add-ons -> Secrets, then attach it to the notebook.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
except Exception:
    pass

os.environ.setdefault("HF_USERNAME", "piyush-mk")
os.environ.setdefault("INVOICEGUARD_CODE_REPO", "piyush-mk/invoiceguard-code")
os.environ.setdefault("BASE_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
# Notebook runs intentionally use notebook-specific repos so they never overwrite HF Jobs outputs.
os.environ.setdefault("HUB_MODEL_ID", "invoiceguard-qwen3-4b-grpo-notebook-smoke-1")
os.environ.setdefault("FORMAT_WARMUP_MODEL_ID", "invoiceguard-qwen3-4b-format-warmup-notebook-smoke-1")
os.environ.setdefault("TRACKIO_PROJECT", "invoiceguard-round2")
os.environ.setdefault("TRACKIO_RUN_NAME", "qwen3-4b-grpo-notebook-smoke-1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN before running training."
print("Configured", os.environ["BASE_MODEL"], "->", f"{os.environ['HF_USERNAME']}/{os.environ['HUB_MODEL_ID']}")
print("Warm-start checkpoint ->", f"{os.environ['HF_USERNAME']}/{os.environ['FORMAT_WARMUP_MODEL_ID']}")


In [ ]:
# Quick GPU sanity check. On CPU this will be too slow for useful training.
import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"gpu memory free/total: {free/1e9:.1f} GB / {total/1e9:.1f} GB")


## Smoke Run

Run this first. It should complete quickly and prove that the environment, model loading, rollout loop, Trackio logging, and Hub artifact upload all work.

In [ ]:
!HF_TOKEN="$HF_TOKEN" \
HF_USERNAME="$HF_USERNAME" \
INVOICEGUARD_CODE_REPO="$INVOICEGUARD_CODE_REPO" \
BASE_MODEL="$BASE_MODEL" \
HUB_MODEL_ID="$HUB_MODEL_ID" \
FORMAT_WARMUP_MODEL_ID="$FORMAT_WARMUP_MODEL_ID" \
TRACKIO_PROJECT="$TRACKIO_PROJECT" \
TRACKIO_RUN_NAME="$TRACKIO_RUN_NAME" \
PYTORCH_CUDA_ALLOC_CONF="$PYTORCH_CUDA_ALLOC_CONF" \
TOKENIZERS_PARALLELISM="$TOKENIZERS_PARALLELISM" \
uv run https://huggingface.co/$INVOICEGUARD_CODE_REPO/resolve/main/training/train_grpo.py \
  --num-iterations 1 \
  --group-size 2 \
  --max-train-tasks 2 \
  --eval-holdout-canonical 1 \
  --eval-holdout-hard 1 \
  --format-warmup-tasks 2 \
  --max-new-tokens 384 \
  --max-prompt-tokens 2048


## SFT (Supervised Fine-Tuning) Run

Alternative to GRPO: trains a LoRA adapter on expert traces generated from environment ground truth. Faster and more stable than RL. Run this to get a quick before/after comparison.

In [ ]:
!HF_TOKEN="$HF_TOKEN" \
HF_USERNAME="$HF_USERNAME" \
INVOICEGUARD_CODE_REPO="$INVOICEGUARD_CODE_REPO" \
BASE_MODEL="$BASE_MODEL" \
HUB_MODEL_ID="invoiceguard-qwen3-4b-sft-notebook-1" \
TRACKIO_PROJECT="$TRACKIO_PROJECT" \
TRACKIO_RUN_NAME="qwen3-4b-sft-notebook-1" \
PYTORCH_CUDA_ALLOC_CONF="$PYTORCH_CUDA_ALLOC_CONF" \
TOKENIZERS_PARALLELISM="$TOKENIZERS_PARALLELISM" \
uv run https://huggingface.co/$INVOICEGUARD_CODE_REPO/resolve/main/training/train_sft.py \
  --hub-model-id invoiceguard-qwen3-4b-sft-notebook-1 \
  --num-epochs 4 \
  --eval-holdout-canonical 3 \
  --eval-holdout-hard 3 \
  --max-new-tokens 384 \
  --max-prompt-tokens 2048

## Full Backup Run

Use this if HF Jobs fails or stalls. It uses the same trainer and pushes a separate adapter repo so we can compare it cleanly against the HF Jobs run.

In [ ]:
# Uncomment to launch a fuller notebook-backed run.
# These names are notebook-specific and will not overwrite HF Jobs outputs.
# os.environ["HUB_MODEL_ID"] = "invoiceguard-qwen3-4b-grpo-notebook-full-1"
# os.environ["FORMAT_WARMUP_MODEL_ID"] = "invoiceguard-qwen3-4b-format-warmup-notebook-full-1"
# os.environ["TRACKIO_RUN_NAME"] = "qwen3-4b-grpo-notebook-full-1"
# !HF_TOKEN="$HF_TOKEN" \
# HF_USERNAME="$HF_USERNAME" \
# INVOICEGUARD_CODE_REPO="$INVOICEGUARD_CODE_REPO" \
# BASE_MODEL="$BASE_MODEL" \
# HUB_MODEL_ID="$HUB_MODEL_ID" \
# FORMAT_WARMUP_MODEL_ID="$FORMAT_WARMUP_MODEL_ID" \
# TRACKIO_PROJECT="$TRACKIO_PROJECT" \
# TRACKIO_RUN_NAME="$TRACKIO_RUN_NAME" \
# PYTORCH_CUDA_ALLOC_CONF="$PYTORCH_CUDA_ALLOC_CONF" \
# TOKENIZERS_PARALLELISM="$TOKENIZERS_PARALLELISM" \
# uv run https://huggingface.co/$INVOICEGUARD_CODE_REPO/resolve/main/training/train_grpo.py \
#   --num-iterations 3 \
#   --group-size 4 \
#   --eval-holdout-canonical 3 \
#   --eval-holdout-hard 3 \
#   --format-warmup-tasks 16 \
#   --max-new-tokens 384 \
#   --max-prompt-tokens 2048


In [ ]:
# Verify that the adapter and training artifacts landed on the Hub.
# Change VERIFY_REPO to whichever run you want to inspect.
VERIFY_REPO = f"{os.environ['HF_USERNAME']}/invoiceguard-qwen3-4b-sft-notebook-1"  # or use HUB_MODEL_ID for GRPO

from huggingface_hub import HfApi, hf_hub_download
import json

api = HfApi(token=os.environ["HF_TOKEN"])
files = api.list_repo_files(repo_id=VERIFY_REPO, repo_type="model")
print(f"Repo: {VERIFY_REPO}")
print(f"Total files: {len(files)}\n")
for path in sorted(files):
    if path.startswith("training_artifacts/") or path in {"adapter_config.json", "adapter_model.safetensors"}:
        print(f"  {path}")

try:
    summary_path = hf_hub_download(repo_id=VERIFY_REPO, filename="training_artifacts/training_summary.json", token=os.environ["HF_TOKEN"])
    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)
    print(f"\n--- Training Summary ---")
    print(json.dumps(summary, indent=2)[:2000])
except Exception as e:
    print(f"\nNo training_summary.json yet: {e}")


## What Judges Should See After A Successful Run

A completed run uploads the following to the Hugging Face Hub:

| Artifact | Location |
|----------|----------|
| LoRA adapter weights | `adapter_model.safetensors` + `adapter_config.json` |
| Training metrics | `training_artifacts/metrics.jsonl` |
| Reward/loss curves | `training_artifacts/*.png` |
| Rollout samples | `training_artifacts/rollout_samples_*.jsonl` |
| Training summary | `training_artifacts/training_summary.json` |

The training summary JSON contains before/after grader scores showing improvement from training.

To use the trained model for inference, merge the LoRA adapter into the base model using `training/merge_adapter.py`, or load it directly with PEFT for evaluation using `training/eval_adapter.py`.